# Day 173 — MLflow Experiment Tracking
## Month 10 | Google Colab

---

### Month 10 Scorecard
| Day | Topic | Score |
|-----|-------|-------|
| 169 | LangChain Chains & Memory | ✅ 80/80+10★ |
| 170 | LangChain Tools & Agents | ✅ 80/80+10★ |
| 171 | Document Loaders + LCEL | ✅ 80/80+10★ |
| 172 | LangChain Capstone | ✅ 90/90+10★ |
| **173** | **MLflow Experiment Tracking** | **← Today** |

---

### Today's Scorecard
| Task | Topic | Points |
|------|-------|--------|
| T1 | Dataset Setup + MLflow Experiment | 15 |
| T2 | Log 3 Runs (params, metrics, tags) | 30 |
| T3 | Programmatic Run Comparison | 20 |
| T4 | Log Artifact (Confusion Matrix) | 15 |
| T5 | NRA Insight from Experiments | 10 |
| ★ | Model Logging via `mlflow.sklearn.log_model()` | 10★ |
| **Total** | | **90/90 + 10★** |

**Dataset:** ReviewPulse India (600 rows, seed=155)
**Target:** `high_rating` (rating ≥ 4) — binary classification
**Environment:** Google Colab

---

### What is MLflow?
MLflow is an open-source platform to manage the ML lifecycle:
- **Tracking** — log parameters, metrics, artifacts per run
- **Projects** — package ML code for reproducibility
- **Models** — deploy models in standard format
- **Registry** — centralized model store with versioning

Today covers **Tracking** (the most used component in client work).

## ⚙️ Cell 0 — Install & Imports

> **After running:** Runtime → Restart runtime → Run All

In [1]:
# Pinned installs
!pip install -q mlflow==2.13.2 scikit-learn pandas numpy matplotlib

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab artifact saving
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import mlflow
import mlflow.sklearn
import os

print(f"MLflow version: {mlflow.__version__}")
print("All imports OK")

MLflow version: 2.13.2
All imports OK


---

## 📦 Section 1 — Raw Data (DO NOT MODIFY)

ReviewPulse India | 600 rows | seed=155

> **Rule:** Copy this cell as-is. Never edit the generation code.

In [2]:
# ── RAW DATA — DO NOT MODIFY ──
np.random.seed(155)
n = 600
categories = ['Electronics', 'Clothing', 'Home & Kitchen', 'Books', 'Sports']
platforms  = ['Amazon', 'Flipkart', 'Myntra', 'Nykaa', 'Meesho']

product_cat      = np.random.choice(categories, n)
platform         = np.random.choice(platforms, n)
sentiment        = np.random.choice(['negative','neutral','positive'], n, p=[0.44, 0.30, 0.26])
rating           = np.where(sentiment=='negative', np.random.randint(1, 3, n),
                   np.where(sentiment=='neutral',  np.random.randint(2, 5, n),
                                                   np.random.randint(3, 6, n)))
review_length    = np.random.randint(50, 500, n)
hired_again      = np.random.choice([0, 1], n, p=[0.65, 0.35])
verified_purchase = np.random.choice([0, 1], n, p=[0.3, 0.7])

df_raw = pd.DataFrame({
    'product_category': product_cat, 'platform': platform,
    'sentiment': sentiment,          'rating': rating,
    'review_length': review_length,  'hired_again': hired_again,
    'verified_purchase': verified_purchase
})
print(f"Raw shape: {df_raw.shape}")
print(df_raw.head(3))

Raw shape: (600, 7)
  product_category  platform sentiment  rating  review_length  hired_again  \
0           Sports     Nykaa  negative       2            145            1   
1         Clothing     Nykaa   neutral       2            402            1   
2      Electronics  Flipkart  positive       3            109            0   

   verified_purchase  
0                  1  
1                  0  
2                  1  


---

## 📖 Section 2 — Concept Notes

### MLflow Core Concepts

| Concept | What it does | Analogy |
|---------|-------------|--------|
| **Experiment** | A named container for related runs | A project folder |
| **Run** | One training execution with logged data | A lab notebook entry |
| **Params** | Hyperparameters (logged once) | Settings you chose |
| **Metrics** | Performance numbers (logged per step/final) | Results you measured |
| **Artifacts** | Files saved to the run (plots, models, CSVs) | Output attachments |
| **Tags** | Key-value metadata | Labels / sticky notes |

### Why it matters in client work
Clients ask: *"Which model version is in production and why?"*
MLflow lets you answer with: run ID, logged params, AUC, and the saved model file — all reproducible.

### Key API pattern
```python
import mlflow

mlflow.set_experiment("My_Experiment")      # create/select experiment

with mlflow.start_run(run_name="run_1"):    # open a run context
    mlflow.log_param("C", 0.1)              # log one hyperparameter
    mlflow.log_params({"model": "LR"})      # log dict of params
    mlflow.log_metric("auc", 0.87)          # log one metric
    mlflow.log_metrics({"acc": 0.83})       # log dict of metrics
    mlflow.set_tag("author", "deepanshu")   # metadata tag
    mlflow.log_artifact("plot.png")         # save a file to the run
    mlflow.sklearn.log_model(model, "model") # save sklearn model
```

### Programmatic querying
```python
runs_df = mlflow.search_runs(experiment_names=["My_Experiment"])
# Returns a pandas DataFrame — sort, filter, compare
best_run = runs_df.sort_values("metrics.auc", ascending=False).iloc[0]
```

---

## ✏️ Section 3 — Practice Tasks

### Feature Engineering (run this first — shared across all tasks)

In [3]:
# Goal: prepare features for classification
# Method: encode sentiment, create binary target high_rating (rating >= 4)

df = df_raw.copy()

# Target: high_rating — binary (1 if rating >= 4, else 0)
df['high_rating'] = (df['rating'] >= 4).astype(int)

# Encode sentiment: negative=0, neutral=1, positive=2
le = LabelEncoder()
df['sentiment_enc'] = le.fit_transform(df['sentiment'])

# Features: sentiment_enc, review_length, verified_purchase
X = df[['sentiment_enc', 'review_length', 'verified_purchase']]
y = df['high_rating']

# Train/test split — stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Target: high_rating | Positive rate: {y.mean()*100:.2f}% ({y.sum()} / {len(y)})")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"LabelEncoder classes: {list(le.classes_)}  (order: {list(le.transform(le.classes_))})")
print(f"\nFeature columns: {list(X.columns)}")

Target: high_rating | Positive rate: 24.67% (148 / 600)
Train: 480, Test: 120
LabelEncoder classes: ['negative', 'neutral', 'positive']  (order: [0, 1, 2])

Feature columns: ['sentiment_enc', 'review_length', 'verified_purchase']


---

### T1 — Dataset Setup + MLflow Experiment (15 pts)

**Tasks:**
- T1a [5 pts]: Set MLflow tracking URI to `./mlruns` (local). Create experiment named `"ReviewPulse_Experiment"`. Print the experiment ID.
- T1b [5 pts]: Print dataset summary: total rows, high_rating count, high_rating %, train size, test size.
- T1c [5 pts]: Print sentiment × high_rating crosstab (show mean high_rating per sentiment group).

In [4]:
# T1a: Set tracking URI and create experiment
# Goal: Configure MLflow to store runs locally and create a named experiment.
# Method: set_tracking_uri -> set_experiment -> fetch and print experiment ID.

import mlflow

mlflow.set_tracking_uri("./mlruns")  # local storage

experiment_name = "ReviewPulse_Experiment"
mlflow.set_experiment(experiment_name)  # creates if not exists

experiment = mlflow.get_experiment_by_name(experiment_name)
print(f"Experiment ID: {experiment.experiment_id}")

2026/06/28 05:48:54 INFO mlflow.tracking.fluent: Experiment with name 'ReviewPulse_Experiment' does not exist. Creating a new experiment.


Experiment ID: 983969990073206209


In [5]:
# T1b: Dataset summary
# Goal: Print descriptive statistics of the dataset.
# Method: Use len(), .sum(), .mean() and print formatted strings.

total_rows = len(df)
high_rating_count = df['high_rating'].sum()
high_rating_pct = df['high_rating'].mean() * 100
train_size = len(X_train)
test_size = len(X_test)

print(f"Total rows: {total_rows}")
print(f"high_rating count: {high_rating_count}")
print(f"high_rating %: {high_rating_pct:.2f}%")
print(f"Train size: {train_size}")
print(f"Test size: {test_size}")

Total rows: 600
high_rating count: 148
high_rating %: 24.67%
Train size: 480
Test size: 120


In [6]:
# T1c: Sentiment × high_rating crosstab (mean high_rating per sentiment)
# Goal: Show how high_rating varies across sentiment categories.
# Method: groupby('sentiment') and compute mean of high_rating.

sentiment_highrating_mean = df.groupby('sentiment')['high_rating'].mean()
print("Mean high_rating per sentiment:")
print(sentiment_highrating_mean)

Mean high_rating per sentiment:
sentiment
negative    0.000000
neutral     0.279330
positive    0.675862
Name: high_rating, dtype: float64


---

### T2 — Log 3 Runs (30 pts | 10 pts each)

Train three models and log each as a separate MLflow run.

**Each run must log:**
- `log_param` / `log_params`: model name + hyperparams
- `log_metrics`: `accuracy`, `f1`, `auc` (all rounded to 4 dp)
- `set_tag`: `"model_type"` → the sklearn class name (e.g. `"LogisticRegression"`)
- `run_name` set in `start_run()`

**Run 1 — LR_C0.1:** `LogisticRegression(C=0.1, max_iter=200, random_state=42)`
**Run 2 — LR_C1.0:** `LogisticRegression(C=1.0, max_iter=200, random_state=42)`
**Run 3 — RF_n50:** `RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)`

After each run, print: `Run ID | Name | AUC`

In [7]:
# T2 Run 1 — LR C=0.1
# Goal: Train LogisticRegression with C=0.1, log hyperparameters, metrics (accuracy, f1, auc), and model tag.
# Method: Use mlflow.start_run with run_name, call log_param(s), log_metric(s), set_tag, then print run info.

with mlflow.start_run(run_name="LR_C0.1") as run:
    model = LogisticRegression(C=0.1, max_iter=200, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    # Round to 4 decimal places
    acc = round(acc, 4)
    f1 = round(f1, 4)
    auc = round(auc, 4)

    # Log params
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("C", 0.1)
    mlflow.log_param("max_iter", 200)
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("auc", auc)

    # Log tag
    mlflow.set_tag("model_type", "LogisticRegression")

    print(f"Run ID: {run.info.run_id} | Name: LR_C0.1 | AUC: {auc}")

Run ID: 26b6b7fb97cf4d09989be20c85746cd0 | Name: LR_C0.1 | AUC: 0.868


In [8]:
# T2 Run 2 — LR C=1.0
# Goal: Train LogisticRegression with C=1.0, log hyperparameters, metrics, tag.
# Method: Same as Run 1, but with C=1.0.

with mlflow.start_run(run_name="LR_C1.0") as run:
    model = LogisticRegression(C=1.0, max_iter=200, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = round(accuracy_score(y_test, y_pred), 4)
    f1 = round(f1_score(y_test, y_pred), 4)
    auc = round(roc_auc_score(y_test, y_prob), 4)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("max_iter", 200)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("auc", auc)

    mlflow.set_tag("model_type", "LogisticRegression")

    print(f"Run ID: {run.info.run_id} | Name: LR_C1.0 | AUC: {auc}")

Run ID: 02a25935ec6c48478c256a7213ed946f | Name: LR_C1.0 | AUC: 0.8657


In [9]:
# T2 Run 3 — RF n=50
# Goal: Train RandomForestClassifier with n_estimators=50, max_depth=5, log all.
# Method: Same structure as previous runs.

with mlflow.start_run(run_name="RF_n50") as run:
    model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = round(accuracy_score(y_test, y_pred), 4)
    f1 = round(f1_score(y_test, y_pred), 4)
    auc = round(roc_auc_score(y_test, y_prob), 4)

    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 50)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("auc", auc)

    mlflow.set_tag("model_type", "RandomForestClassifier")

    print(f"Run ID: {run.info.run_id} | Name: RF_n50 | AUC: {auc}")

Run ID: 1a72db2abe9744e5b4d0927ead60fcde | Name: RF_n50 | AUC: 0.8674


---

### T3 — Programmatic Run Comparison (20 pts)

**Tasks:**
- T3a [8 pts]: Call `mlflow.search_runs(experiment_names=["ReviewPulse_Experiment"])` to get a DataFrame. Print all rows showing: `run_id`, `tags.mlflow.runName`, `metrics.accuracy`, `metrics.f1`, `metrics.auc` — sorted by `metrics.auc` descending.
- T3b [6 pts]: Extract and print: `best_run_id`, `best_run_name`, `best_auc` from the DataFrame.
- T3c [6 pts]: Print a formatted comparison table:
  ```
  Run Name    | Accuracy | F1     | AUC
  ------------|----------|--------|------
  LR_C0.1     | 0.8250   | 0.6182 | 0.8680
  ...
  ```

In [10]:
# T3a: search_runs and display
# Goal: Retrieve all runs from the experiment as a DataFrame, sort by AUC descending, print run_id, run name, metrics.
# Method: mlflow.search_runs with experiment_names, then filter and sort columns.

runs_df = mlflow.search_runs(experiment_names=["ReviewPulse_Experiment"])
# Select relevant columns
cols = ['run_id', 'tags.mlflow.runName', 'metrics.accuracy', 'metrics.f1', 'metrics.auc']
# Sort by AUC descending
runs_df_sorted = runs_df[cols].sort_values('metrics.auc', ascending=False)

print("All runs sorted by AUC (descending):")
print(runs_df_sorted.to_string(index=False))

All runs sorted by AUC (descending):
                          run_id tags.mlflow.runName  metrics.accuracy  metrics.f1  metrics.auc
26b6b7fb97cf4d09989be20c85746cd0             LR_C0.1             0.825      0.6182       0.8680
1a72db2abe9744e5b4d0927ead60fcde              RF_n50             0.800      0.5385       0.8674
02a25935ec6c48478c256a7213ed946f             LR_C1.0             0.825      0.6182       0.8657


In [11]:
# T3b: extract best run details
# Goal: Identify the run with highest AUC and print its id, name, and AUC.
# Method: Use .iloc[0] on sorted DataFrame.

best_row = runs_df_sorted.iloc[0]
best_run_id = best_row['run_id']
best_run_name = best_row['tags.mlflow.runName']
best_auc = best_row['metrics.auc']

print(f"Best run ID: {best_run_id}")
print(f"Best run name: {best_run_name}")
print(f"Best AUC: {best_auc}")

Best run ID: 26b6b7fb97cf4d09989be20c85746cd0
Best run name: LR_C0.1
Best AUC: 0.868


In [12]:
# T3c: formatted comparison table
# Goal: Print a clean table with run name, accuracy, f1, AUC.
# Method: Iterate over rows and use string formatting.

print("\nRun Name    | Accuracy | F1     | AUC")
print("------------|----------|--------|------")
for _, row in runs_df_sorted.iterrows():
    name = row['tags.mlflow.runName']
    acc = row['metrics.accuracy']
    f1 = row['metrics.f1']
    auc = row['metrics.auc']
    print(f"{name:<11} | {acc:.4f} | {f1:.4f} | {auc:.4f}")


Run Name    | Accuracy | F1     | AUC
------------|----------|--------|------
LR_C0.1     | 0.8250 | 0.6182 | 0.8680
RF_n50      | 0.8000 | 0.5385 | 0.8674
LR_C1.0     | 0.8250 | 0.6182 | 0.8657


---

### T4 — Log Artifact: Confusion Matrix (15 pts)

For the **best model** (LR C=0.1):

- T4a [6 pts]: Retrain it, get predictions on test set, compute `confusion_matrix(y_test, y_pred)`.
  Print: TN, FP, FN, TP from `.ravel()`.
- T4b [9 pts]: Create a confusion matrix heatmap plot (use `matplotlib`, dark blue `#1F3864` color scheme), save as `confusion_matrix.png`, then open a **new MLflow run** named `"Best_Model_Artifacts"` and log the PNG via `mlflow.log_artifact("confusion_matrix.png")`.
  Print: `Artifact logged to run: {run_id}`

In [13]:
# T4a: Retrain best model, confusion matrix values
# Goal: Train LogisticRegression(C=0.1) on the full training set, predict test, get confusion matrix, and print components.
# Method: Use confusion_matrix and .ravel().

# Best model from T2 is LR C=0.1
best_model = LogisticRegression(C=0.1, max_iter=200, random_state=42)
best_model.fit(X_train, y_train)
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion matrix values:")
print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")

Confusion matrix values:
TN: 82, FP: 8, FN: 13, TP: 17


In [14]:
# T4b: Plot, save, log artifact
# Goal: Generate a heatmap of the confusion matrix, save as 'confusion_matrix.png', and log it in a new MLflow run.
# Method: Use matplotlib to plot, savefig, then mlflow.start_run and mlflow.log_artifact.

import matplotlib.pyplot as plt
import seaborn as sns  # (optional but useful; can also use imshow)

# Create the heatmap
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
plt.title('Confusion Matrix (Best Model: LR C=0.1)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.close()

# Now log this artifact in a new run
with mlflow.start_run(run_name="Best_Model_Artifacts") as run:
    mlflow.log_artifact("confusion_matrix.png")
    print(f"Artifact logged to run: {run.info.run_id}")

Artifact logged to run: 01b19730fa5e4c71b5ddf63001cb642e


---

### T5 — NRA Insight from Experiments (10 pts)

Write **1 NRA bullet** based on the experiment results.

**Rules:**
- **Number** → read directly from a printed cell output above (e.g. best AUC, accuracy, feature importance)
- **Reason** → state a causal mechanism (WHY does this pattern exist in the data?)
- **Action** → specific and committed — name the model/threshold/parameter, no hedging

**Template:**
```
N: [metric and value from output]
R: [causal mechanism — why this model / result occurred]
A: [specific deployment/next-step recommendation]
```

In [18]:
# T5: NRA Insight
# Goal: Provide a Number–Reason–Action bullet using metrics from printed outputs.
# The numbers must come from the actual output; here we reference the best AUC and accuracy.

nra_insight_final = """
N: Best model (LR C=0.1) achieved AUC = 0.8680 and accuracy = 0.8250 on the test set, outperforming LR C=1.0 (AUC = 0.8657) and RF_n50 (AUC = 0.8674).

R: Since C is the inverse of regularization strength, C=0.1 applies stronger L2 regularization than C=1.0. This stronger penalty constrains the coefficient magnitudes on this small 3-feature set (sentiment_enc, review_length, verified_purchase), effectively reducing variance without sacrificing discrimination along the linear sentiment → rating boundary, which is why AUC improves over the weaker-regularization C=1.0 model. The random forest with limited depth (5) and trees (50) fails to capture additional nonlinear interactions in this small dataset, resulting in a slightly lower AUC.

A: Deploy LogisticRegression(C=0.1) as the production classifier. In the next iteration, conduct a grid search over C in [0.05, 0.5] and monitor calibration, because the current confusion matrix (TP=17, FP=8) shows a good precision/recall trade-off that finer tuning could further optimize.
"""
print(nra_insight_final)


N: Best model (LR C=0.1) achieved AUC = 0.8680 and accuracy = 0.8250 on the test set, outperforming LR C=1.0 (AUC = 0.8657) and RF_n50 (AUC = 0.8674).

R: Since C is the inverse of regularization strength, C=0.1 applies stronger L2 regularization than C=1.0. This stronger penalty constrains the coefficient magnitudes on this small 3-feature set (sentiment_enc, review_length, verified_purchase), effectively reducing variance without sacrificing discrimination along the linear sentiment → rating boundary, which is why AUC improves over the weaker-regularization C=1.0 model. The random forest with limited depth (5) and trees (50) fails to capture additional nonlinear interactions in this small dataset, resulting in a slightly lower AUC.

A: Deploy LogisticRegression(C=0.1) as the production classifier. In the next iteration, conduct a grid search over C in [0.05, 0.5] and monitor calibration, because the current confusion matrix (TP=17, FP=8) shows a good precision/recall trade-off that 

---

### ★ Bonus — Log Model to MLflow Registry (10★)

For the best model (LR C=0.1):
- Open a new run named `"Best_Model_Registry"`
- Use `mlflow.sklearn.log_model(model, artifact_path="model", registered_model_name="ReviewPulse_Classifier")` to log AND register it
- Print: `Registered model name`, `model URI`

> **Note:** With local tracking, the model is saved in `./mlruns`. On Databricks or a remote MLflow server, this becomes a full versioned registry.

In [16]:
# ★ Bonus: mlflow.sklearn.log_model + register
# Goal: Save the best model (LR C=0.1) as an MLflow artifact and register it with a model name.
# Method: Open a new run, call log_model with registered_model_name.

with mlflow.start_run(run_name="Best_Model_Registry") as run:
    # Retrain or reuse the best model (we can reuse best_model from T4a)
    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
        registered_model_name="ReviewPulse_Classifier"
    )
    # Model URI: runs:/<run_id>/model
    model_uri = f"runs:/{run.info.run_id}/model"
    print(f"Registered model name: ReviewPulse_Classifier")
    print(f"Model URI: {model_uri}")

Registered model name: ReviewPulse_Classifier
Model URI: runs:/150590115fb6447c9d0b24911de923a4/model


Successfully registered model 'ReviewPulse_Classifier'.
Created version '1' of model 'ReviewPulse_Classifier'.


---

## 📊 Section 4 — Scoring Rubric

| Task | Sub-task | Points | What to check |
|------|----------|--------|----------------|
| **T1** | T1a: Tracking URI set + experiment created + ID printed | 5 | `mlflow.set_tracking_uri`, `mlflow.create_experiment` or `set_experiment` |
| | T1b: All 5 summary values printed (rows, count, %, train, test) | 5 | Values match locked key |
| | T1c: Sentiment × high_rating crosstab — correct 3 values | 5 | negative=0.000, neutral≈0.279, positive≈0.676 |
| **T2** | Run1 LR_C0.1 — params + metrics + tag logged, AUC printed | 10 | `with mlflow.start_run` pattern; metrics match |
| | Run2 LR_C1.0 — same requirements | 10 | |
| | Run3 RF_n50 — same requirements; n_estimators=50, max_depth=5 | 10 | |
| **T3** | T3a: search_runs DataFrame printed, sorted by AUC desc | 8 | All 3 runs visible; correct columns |
| | T3b: best_run_id, best_run_name, best_auc extracted and printed | 6 | best_run_name=LR_C0.1, auc=0.8680 |
| | T3c: Formatted comparison table | 6 | All 3 rows, correct alignment |
| **T4** | T4a: TN=82, FP=8, FN=13, TP=17 printed | 6 | From `.ravel()` |
| | T4b: PNG saved + logged as artifact in new run | 9 | `log_artifact` called; run ID printed |
| **T5** | NRA bullet — Number from output, Reason causal, Action specific | 10 | Standard NRA rubric |
| **★** | log_model + registered_model_name set; model URI printed | 10★ | `mlflow.sklearn.log_model` with registry |
| **TOTAL** | | **90 + 10★** | |

### NRA Deduction Guide
| Error | Deduction |
|-------|-----------|
| Number not from printed output (estimated or remembered) | −3 |
| Reason is outcome description, not causal mechanism | −2 |
| Action uses hedging language ("might", "could consider") | −2 |
| Action doesn't name specific model/threshold | −2 |

---

### Interview Answer
**Q: Why do data scientists use MLflow instead of just printing metrics?**

*"Without MLflow, metric logging is scattered across print statements and notebooks — reproducibility breaks the moment you close a session. MLflow creates a structured, queryable record of every experiment: the exact hyperparameters, the metric values, the artifact files, and the model binary. When a client asks 'which model is in production and why did we choose it', MLflow lets me point to a run ID with logged evidence rather than saying 'I think it was that notebook from last week'. The real value is comparability — `mlflow.search_runs()` gives you all experiments as a DataFrame you can sort and filter instantly."*

---

**GitHub commit:**
```
feat: Day173 - MLflow Experiment Tracking [pending]
```
Repo: `Month10-LangChain-MLflow-Portfolio`